# Confronto Random Forest vs VQC vs QSVC sul dataset CICIDS2017

Questo notebook documenta in stile tutorial l'addestramento e la valutazione di **quattro modelli** di machine learning sul dataset **CICIDS2017** per la classificazione binaria del traffico di rete (BENIGN vs ATTACK):

1. **Random Forest** — baseline classica di riferimento (paper Sharafaldin et al., 2018)
2. **VQC** con ansatz `real_amplitudes` — classificatore variazionale quantistico
3. **VQC** con ansatz `efficient_su2` — variante più espressiva del VQC
4. **QSVC** con `FidelityStatevectorKernel` — Support Vector Classifier che usa un kernel calcolato quantisticamente

L'obiettivo è verificare se alcuni approcci di **quantum machine learning** (QML) siano competitivi rispetto al miglior classificatore classico riportato nel paper di riferimento del dataset:

> Sharafaldin I., Habibi Lashkari A., Ghorbani A. A. (2018), *Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization*, ICISSP — F1-score Random Forest = 0.97.

La struttura del notebook è la seguente:

1. Analisi esplorativa del dataset
2. Preprocessing e pulizia
3. Sottocampionamento bilanciato
4. Feature selection engineering: scaling, selezione e riduzione dimensionale
5. Split train/test
6. Addestramento del Random Forest
7. Valutazione del modello
8. Confronto RF su feature complete vs RF su feature ridotte
9. Preparazione dell'input per i modelli quantistici
10. Import e setup di Qiskit
11. VQC con ansatz `real_amplitudes`
12. VQC con ansatz `efficient_su2`
13. QSVC — Quantum Support Vector Classifier
14. Confronto finale tra tutti i modelli
15. Conclusioni complessive


## 1. Analisi esplorativa del dataset

Il dataset **CICIDS2017** è distribuito dal [Canadian Institute for Cybersecurity (CIC)](http://www.unb.ca/cic/datasets/IDS2017.html) come cartella `MachineLearningCVE` contenente otto file CSV giornalieri. Ogni file raccoglie i flussi di rete catturati in un giorno lavorativo, descritti da circa 80 feature numeriche estratte tramite *CICFlowMeter* (durata del flusso, byte totali, lunghezza media dei pacchetti, flag TCP, ecc.) e corredati da una colonna `Label` contenente l'etichetta della classe: `BENIGN` oppure uno dei ~14 tipi di attacco (DoS, DDoS, PortScan, Brute Force, Infiltration, Botnet, …).

CICIDS2017 presenta caratteristiche reali di un problema di *intrusion detection*:

- è **grande** (oltre 2,8 milioni di flussi),
- è fortemente **sbilanciato** verso la classe BENIGN,
- contiene **righe sporche** (valori `+inf`, `-inf`, `NaN`) dovute a divisioni per zero nel calcolo delle feature,
- ha **spazi** indesiderati nei nomi delle colonne (bug noto del dataset).

Per prima cosa importiamo le librerie necessarie e fissiamo il seed per la riproducibilità.

In [ ]:
import os
import glob
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

### 1.1 Caricamento dei CSV

Definiamo una funzione `load_cicids2017` che concatena tutti i file CSV presenti nella cartella `DATA_PATH` in un unico `DataFrame` pandas. La funzione applica due piccole ma fondamentali operazioni di pulizia sul momento:

- usa `encoding="latin1"` perché alcune righe contengono caratteri non UTF-8 che farebbero fallire la lettura;
- rimuove gli spazi nei nomi delle colonne con `str.strip();

In [ ]:
DATA_PATH = "./archive/"   # cartella contenente i CSV di CICIDS2017

def load_cicids2017(data_path: str) -> pd.DataFrame:
    """Carica e concatena tutti i CSV del CICIDS2017 presenti in data_path."""
    csv_files = sorted(glob.glob(os.path.join(data_path, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(
            f"Nessun file CSV trovato in '{data_path}'. "
            "Scaricare il dataset da http://www.unb.ca/cic/datasets/IDS2017.html"
        )
    dfs = []
    for f in csv_files:
        print(f"    - carico {os.path.basename(f)}")
        dfs.append(pd.read_csv(f, encoding="latin1", low_memory=False))
    df = pd.concat(dfs, ignore_index=True)
    df.columns = df.columns.str.strip()   # rimuove gli spazi dai nomi colonna
    return df

df = load_cicids2017(DATA_PATH)
print(f"\nTotale flussi : {len(df):,}")
print(f"Colonne       : {df.shape[1]}")

### 1.2 Distribuzione delle classi

Diamo uno sguardo alla distribuzione della colonna `Label` per confermare lo sbilanciamento tra classi tipico dei dataset di intrusion detection.

In [ ]:
print(df["Label"].value_counts())

Come atteso, la classe `BENIGN` domina il dataset (oltre l'80% dei flussi), mentre gli attacchi sono distribuiti in modo molto disomogeneo tra le 14 categorie. Questo sbilanciamento dovrà essere trattato nelle fasi successive, sia riducendo il problema a binario (BENIGN vs ATTACK), sia tramite un sottocampionamento bilanciato.

## 2. Preprocessing

La fase di preprocessing svolge tre compiti:

1. **Pulizia**: sostituisce i valori `+inf` e `-inf` con `NaN` e rimuove tutte le righe che contengono `NaN`. Questi valori nascono da divisioni per zero nel calcolo di feature come *packet length mean* o *flow bytes per second* quando la durata del flusso è nulla.
2. **Binarizzazione dell'etichetta**: per il confronto con il VQC riformuliamo il problema come classificazione binaria (`BENIGN` vs `ATTACK`), coerentemente con la scelta più diffusa in letteratura quando si fa quantum ML su CICIDS2017 (il VQC su simulatore non riesce a gestire molte classi).
3. **Selezione delle colonne numeriche**: scartiamo eventuali colonne non numeriche (ad esempio timestamp testuali) tramite `select_dtypes(include=[np.number])`.

Infine usiamo `LabelEncoder` per convertire le etichette testuali `ATTACK`/`BENIGN` in interi `0`/`1`.

In [ ]:
def preprocess(df: pd.DataFrame, binary: bool = True):
    """Pulisce il dataset e restituisce X, y, LabelEncoder e lista feature."""
    label_col = "Label"
    if label_col not in df.columns:
        raise KeyError(f"Colonna '{label_col}' non trovata.")

    # 1. rimozione di +inf / -inf / NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    before = len(df)
    df = df.dropna()
    print(f"    righe rimosse per NaN/Inf: {before - len(df):,} "
          f"({(before - len(df)) / before * 100:.2f}%)")

    # 2. binarizzazione dell'etichetta
    if binary:
        df = df.copy()
        df[label_col] = df[label_col].apply(
            lambda v: "BENIGN" if str(v).strip().upper() == "BENIGN" else "ATTACK"
        )

    # 3. solo colonne numeriche come feature
    y_raw = df[label_col].values
    X_df  = df.drop(columns=[label_col]).select_dtypes(include=[np.number])

    le = LabelEncoder()
    y  = le.fit_transform(y_raw)

    return X_df.values.astype(np.float64), y, le, X_df.columns.tolist()

X, y, le, feat_names = preprocess(df, binary=True)
print(f"Shape dopo pulizia : {X.shape}")
counts = {cls: int((y == i).sum()) for i, cls in enumerate(le.classes_)}
print(f"Distribuzione      : {counts}")

## 3. Sottocampionamento bilanciato

Anche dopo la binarizzazione, la classe `BENIGN` resta nettamente dominante. Addestrare un classificatore su un dataset così sbilanciato porta a modelli che massimizzano l'accuracy *ignorando la classe minoritaria* (il classificatore triviale "tutto BENIGN" otterrebbe già ~80% di accuracy).

Per il confronto con il VQC adottiamo quindi un **sottocampionamento bilanciato**: estraiamo lo stesso numero di flussi per ciascuna classe fino a raggiungere un totale di `SAMPLE_SIZE`. Questa è anche una necessità pratica: il VQC su simulatore statevector ha un tempo di addestramento per campione molto superiore al Random Forest, quindi dobbiamo limitarci a qualche migliaio di flussi.

Usiamo il moderno `np.random.default_rng` con seed fisso per avere la stessa estrazione a ogni esecuzione.

In [ ]:
SAMPLE_SIZE = 2000   # numero totale di flussi dopo il bilanciamento

def balanced_subsample(X, y, n_total, seed=42):
    """Estrae lo stesso numero di sample per ogni classe."""
    rng       = np.random.default_rng(seed)
    classes   = np.unique(y)
    per_class = n_total // len(classes)

    idx_list = []
    for c in classes:
        idx_c = np.where(y == c)[0]
        take  = min(per_class, len(idx_c))
        idx_list.append(rng.choice(idx_c, size=take, replace=False))

    idx = np.concatenate(idx_list)
    rng.shuffle(idx)
    return X[idx], y[idx]

X, y = balanced_subsample(X, y, SAMPLE_SIZE, seed=RANDOM_SEED)
counts = {cls: int((y == i).sum()) for i, cls in enumerate(le.classes_)}
print(f"Distribuzione post-sampling : {counts}")
print(f"Shape                       : {X.shape}")

## 4. Feature selection engineering

Il Random Forest, in linea di principio, è **invariante a trasformazioni monotone** delle feature: gli alberi decisionali si basano su soglie che non cambiano se riscaliamo le feature. Quindi, strettamente parlando, `StandardScaler` e PCA non sarebbero necessari per addestrare la baseline RF.

Tuttavia, lo scopo di questo lavoro è un **confronto equo tra RF e VQC**. Il VQC richiede un numero di qubit pari al numero di feature e i simulatori classici riescono a gestire praticamente solo fino a 8-12 qubit; dobbiamo dunque ridurre la dimensionalità. Per non penalizzare il RF su feature selezionate male, applichiamo la stessa pipeline a entrambi i modelli:

1. **StandardScaler**: standardizza ogni feature a media 0 e varianza 1.
2. **Feature selection con RF**: addestriamo un Random Forest preliminare su tutte le feature e selezioniamo le top 20 in base alla `feature_importances_`.
3. **PCA**: riduciamo a `N_COMPONENTS = 8` componenti principali (= numero di qubit del VQC).

> **Nota**: nello script originale, dopo la PCA si applica un `MinMaxScaler(feature_range=(0, π))` per mappare le feature nell'intervallo utile allo `ZZFeatureMap` del VQC. Per il **solo** Random Forest questo passaggio è inutile (gli alberi non se ne accorgerebbero), ma lo manteniamo comunque così che i due modelli vedano esattamente lo stesso input.

In [ ]:
# 4a. StandardScaler
X = StandardScaler().fit_transform(X)
print(f"Dopo StandardScaler : media ≈ {X.mean():.2e}, std ≈ {X.std():.2f}")

### 4.1 Feature selection via RF importance

Alleniamo un Random Forest preliminare per calcolare l'importanza di ciascuna feature (riduzione media dell'impurità nei nodi). Teniamo le 20 feature più informative.

In [ ]:
rf_sel = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1
)
rf_sel.fit(X, y)

top_idx = np.argsort(rf_sel.feature_importances_)[-20:]
X = X[:, top_idx]
print(f"Feature selezionate : {len(top_idx)}")
print(f"Shape dopo selezione: {X.shape}")

### 4.2 Riduzione dimensionale con PCA

Salviamo una copia delle feature *prima* della PCA (`X_before_pca`): ci servirà in seguito per addestrare un RF "completo" (su 20 feature) come termine di paragone ideale.

Poi applichiamo la PCA a `N_COMPONENTS = 8`, che è anche il numero di qubit del VQC nella fase successiva. La varianza spiegata dalla PCA ci dice quanta informazione originale preserviamo.

In [ ]:
N_COMPONENTS = 8

X_before_pca = X.copy()   # salvato per il confronto "RF su feature complete"

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_SEED)
X   = pca.fit_transform(X)

print(f"Varianza spiegata  : {pca.explained_variance_ratio_.sum():.3f}")
print(f"Shape dopo PCA     : {X.shape}")

## 5. Split train / test

Dividiamo il dataset ridotto in 80% training e 20% test, stratificando per preservare il bilanciamento tra le classi anche dopo lo split (`stratify=y`). Fissiamo `random_state` al seed globale per la riproducibilità.

In [ ]:
TRAIN_SIZE = 0.8

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,
)
print(f"Train : {X_train.shape} | Test : {X_test.shape}")

## 6. Addestramento del Random Forest

Passiamo all'addestramento del Random Forest. Usiamo l'implementazione di scikit-learn con i seguenti iperparametri:

| Iperparametro | Valore | Motivazione |
|---|---|---|
| `n_estimators` | 300 | Numero di alberi nella foresta. 300 è un valore tipico per problemi IDS: sufficiente per stabilizzare la varianza senza eccessivo costo computazionale. |
| `max_depth` | 20 | Profondità massima di ogni albero. Limita l'overfitting su un dataset relativamente piccolo (2000 flussi dopo sottocampionamento). |
| `class_weight` | `"balanced"` | Ripondera automaticamente le classi in modo inverso alla loro frequenza. Nel nostro caso il dataset è già bilanciato, ma è una rete di sicurezza. |
| `random_state` | `RANDOM_SEED` | Riproducibilità. |
| `n_jobs` | -1 | Usa tutti i core disponibili per l'addestramento parallelo degli alberi. |

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

t0 = time.time()
rf.fit(X_train, y_train)
train_time = time.time() - t0

print(f"Tempo di addestramento : {train_time:.2f} s")

## 7. Valutazione del modello

Adottiamo le quattro metriche usate dal paper originale di CICIDS2017:

- **Accuracy** — frazione di predizioni corrette.
- **Precision** (weighted) — quanti flussi predetti come ATTACK sono veri ATTACK, pesato per la numerosità delle classi.
- **Recall** (weighted) — quanti ATTACK reali sono stati correttamente identificati.
- **F1-score** (weighted) — media armonica di precision e recall.

Il flag `average="weighted"` pesa le metriche in base al supporto di ciascuna classe: è la scelta più onesta su dataset sbilanciati. `zero_division=0` evita avvisi quando una classe non è mai predetta.

Definiamo una funzione `evaluate_model` che può essere riutilizzata anche per il VQC e per l'RF "completo" della sezione successiva.

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test,
                   name, is_quantum=False):
    """Addestra il modello (se non già fit), calcola metriche e tempi."""
    print(f"\n>> {name}")

    # fit (se il modello non è stato già addestrato, ri-addestra)
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - t0

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec  = recall_score   (y_test, y_pred, average="weighted", zero_division=0)
    f1   = f1_score       (y_test, y_pred, average="weighted", zero_division=0)

    print(f"   train time : {train_time:8.2f} s")
    print(f"   pred  time : {pred_time:8.2f} s")
    print(f"   train acc  : {train_acc:6.3f}")
    print(f"   test  acc  : {test_acc:6.3f}")
    print(f"   precision  : {prec:6.3f}")
    print(f"   recall     : {rec:6.3f}")
    print(f"   F1         : {f1:6.3f}")

    return dict(
        name=name,
        type="Quantum" if is_quantum else "Classical",
        train_time=train_time, pred_time=pred_time,
        train_acc=train_acc, test_acc=test_acc,
        precision=prec, recall=rec, f1=f1,
    )

results = []
results.append(
    evaluate_model(rf, X_train, y_train, X_test, y_test,
                   "RandomForest (PCA 8)", is_quantum=False)
)

Il Random Forest raggiunge tipicamente F1 > 0.95 anche su sole 8 componenti PCA: il motivo è che i primi componenti principali catturano gran parte della varianza delle 20 feature selezionate, e gli alberi riescono a sfruttare combinazioni non lineari di queste componenti.

Il *train time* è dell'ordine del secondo: un confronto interessante, perché come vedremo il VQC su simulatore impiegherà diversi minuti sullo stesso dataset.

## 8. Confronto: RF su feature complete

Per capire quanto la PCA penalizza il modello classico, addestriamo un secondo Random Forest sulle **20 feature selezionate** (prima della PCA). Questo è il *limite superiore* del Random Forest in questa pipeline: se nemmeno con 20 feature ben scelte l'RF migliora significativamente, significa che la PCA non è il collo di bottiglia.

Nel paper originale l'RF opera su tutte le ~80 feature e raggiunge F1 = 0.97 in circa 74 s. Qui lo confrontiamo in condizioni più "magre".

In [ ]:
X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
    X_before_pca, y,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,
)

rf_full = RandomForestClassifier(
    n_estimators=300, max_depth=20,
    class_weight="balanced",
    random_state=RANDOM_SEED, n_jobs=-1,
)

results.append(
    evaluate_model(rf_full, X_tr_full, y_tr_full, X_te_full, y_te_full,
                   f"RF ({X_before_pca.shape[1]} feature)",
                   is_quantum=False)
)

## 9. Preparazione dell'input per i modelli quantistici

I modelli quantistici (sia VQC che QSVC) usano lo `ZZFeatureMap` per codificare i dati classici in stati quantistici. Questo circuito applica rotazioni del tipo $e^{i\,\phi_i\,Z_i}$ e $e^{i\,\phi_{ij}\,Z_iZ_j}$, in cui le ampiezze $\phi$ sono **proprio i valori delle feature**. Se le feature non sono limitate in un intervallo ragionevole, le rotazioni diventano sostanzialmente casuali (il modulo $2\pi$ "wrappa" tutto) e il modello non riesce ad apprendere.

La pratica standard è quindi mappare le feature in un intervallo bounded, tipicamente $[0, \pi]$ o $[0, 2\pi]$. Qui scegliamo $[0, \pi]$, valore conservativo che evita aliasing nelle rotazioni.

> **Importante**: il `MinMaxScaler` viene **fittato solo sul training set** (per evitare data leakage dal test set) e poi applicato sia a `X_train` che a `X_test`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Riscaliamo X_train e X_test (entrambi dopo la PCA) in [0, pi].
# Fit solo su train -> niente data leakage.
quantum_scaler = MinMaxScaler(feature_range=(0, np.pi))
Xq_train = quantum_scaler.fit_transform(X_train)
Xq_test  = quantum_scaler.transform(X_test)

print(f"Xq_train  shape : {Xq_train.shape}  range : "
      f"[{Xq_train.min():.3f}, {Xq_train.max():.3f}]")
print(f"Xq_test   shape : {Xq_test.shape}   range : "
      f"[{Xq_test.min():.3f}, {Xq_test.max():.3f}]")

## 10. Import e setup di Qiskit

Importiamo le classi necessarie da `qiskit` e `qiskit_machine_learning`. Fissiamo anche il seed globale dell'algoritmo per la riproducibilità.

In [ ]:
from qiskit.circuit.library import zz_feature_map, real_amplitudes, efficient_su2
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals

algorithm_globals.random_seed = RANDOM_SEED

### 10.1 Lo ZZFeatureMap

Costruiamo lo `ZZFeatureMap` con `feature_dimension = N_COMPONENTS = 8`, che corrisponde al numero di qubit del circuito (un qubit per ogni componente PCA). Il parametro `reps` controlla quante volte la struttura base viene ripetuta: valori alti aumentano l'espressività ma anche la profondità del circuito e il rischio di *barren plateaus*. Manteniamo `reps=1` per limitare la profondità.

In [ ]:
REPS_FEATURE_MAP = 1

num_features = Xq_train.shape[1]   # = N_COMPONENTS = 8

feature_map = zz_feature_map(
    feature_dimension=num_features,
    reps=REPS_FEATURE_MAP,
)
print(f"Feature map : {feature_map.name}")
print(f"  qubit       : {feature_map.num_qubits}")
print(f"  parametri   : {feature_map.num_parameters}")
print(f"  profondità  : {feature_map.decompose().depth()}")

### 10.2 Factory per il VQC

Costruire un VQC richiede sempre gli stessi cinque elementi: sampler, feature map, ansatz, ottimizzatore, callback. Per non duplicare codice, definiamo una piccola factory `build_vqc` che riceve l'ansatz e una lista in cui salvare la storia della funzione obiettivo (utile per il grafico di convergenza).

In [ ]:
MAX_ITER_VQC = 300

def build_vqc(feature_map, ansatz, history_list, label):
    """Costruisce un VQC con callback che logga ogni 10 iterazioni."""
    def _cb(weights, obj_val):
        history_list.append(obj_val)
        if len(history_list) % 10 == 0:
            print(f"   [{label}] iter {len(history_list):3d} | obj = {obj_val:.4f}")
    return VQC(
        sampler=StatevectorSampler(),
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=MAX_ITER_VQC),
        callback=_cb,
    )

## 11. VQC con ansatz `real_amplitudes`

Il primo VQC che addestriamo usa l'ansatz `real_amplitudes`: una sequenza di rotazioni $R_y$ alternate a entanglement CX in pattern *reverse-linear*. È l'ansatz più semplice e parsimonioso disponibile in Qiskit ed è quello suggerito dal tutorial ufficiale Iris come punto di partenza.

Numero di parametri = `num_qubits × (reps + 1) = 8 × 4 = 32`.

> **Avvertenza**: il training di un VQC su simulatore statevector non è veloce. Su un dataset di 1600 sample (80% di 2000) e 8 qubit, una singola iterazione richiede già qualche secondo, e con `MAX_ITER_VQC = 300` il tempo totale può facilmente raggiungere le decine di minuti. È normale.

In [ ]:
REPS_ANSATZ = 3

history_ra = []
ansatz_ra  = real_amplitudes(num_qubits=num_features, reps=REPS_ANSATZ)
vqc_ra     = build_vqc(feature_map, ansatz_ra, history_ra, "real_amp")

print(f"Ansatz : {ansatz_ra.name}")
print(f"  parametri : {ansatz_ra.num_parameters}")

Riusiamo la funzione `evaluate_model` definita nella sezione 7. Il flag `is_quantum=True` cambia solo la colonna `Tipo` nei risultati.

In [ ]:
results.append(
    evaluate_model(vqc_ra, Xq_train, y_train, Xq_test, y_test,
                   "VQC (real_amplitudes)", is_quantum=True)
)

## 12. VQC con ansatz `efficient_su2`

Il secondo VQC usa `efficient_su2`, che differisce da `real_amplitudes` per un dettaglio importante: a ogni qubit applica **sia** $R_y$ **sia** $R_z$. Questo significa coprire tutto il gruppo $SU(2)$ per ciascun qubit, contro la sola rotazione $R_y$ del precedente ansatz. Il prezzo è il **doppio** dei parametri:

`num_qubits × 2 × (reps + 1) = 8 × 2 × 4 = 64 parametri`.

Più parametri = più espressività, ma anche training più lungo e maggior rischio di overfitting su dataset piccoli.

In [ ]:
history_eff = []
ansatz_eff  = efficient_su2(num_qubits=num_features, reps=REPS_ANSATZ)
vqc_eff     = build_vqc(feature_map, ansatz_eff, history_eff, "eff_su2")

print(f"Ansatz : {ansatz_eff.name}")
print(f"  parametri : {ansatz_eff.num_parameters}")

results.append(
    evaluate_model(vqc_eff, Xq_train, y_train, Xq_test, y_test,
                   "VQC (efficient_su2)", is_quantum=True)
)

### 12.1 Confronto della convergenza

Plottiamo le storie della funzione obiettivo dei due VQC sullo stesso grafico. Una curva che decresce e poi si stabilizza indica buona convergenza; una curva piatta sin dall'inizio segnala un *barren plateau* (gradienti esponenzialmente piccoli, problema noto dei VQC con molti qubit / molti layer).

In [ ]:
plt.figure(figsize=(10, 5))
if history_ra:
    plt.plot(history_ra,  label="VQC real_amplitudes",  color="steelblue")
if history_eff:
    plt.plot(history_eff, label="VQC efficient_su2",    color="darkorange")

plt.xlabel("Iterazione COBYLA")
plt.ylabel("Funzione obiettivo (cross-entropy)")
plt.title("Convergenza dei VQC su CICIDS2017")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 13. QSVC — Quantum Support Vector Classifier

Il QSVC è concettualmente molto diverso dal VQC. Anziché ottimizzare i parametri di un circuito variazionale, addestra una **SVM classica** ma con una matrice kernel calcolata quantisticamente. Il kernel quantistico usato qui è la *fidelity* tra gli stati prodotti dalla feature map:

$$
K(\vec{x}_i, \vec{x}_j) = \bigl|\langle \phi(\vec{x}_i) \,|\, \phi(\vec{x}_j) \rangle \bigr|^2.
$$

| Aspetto | VQC | QSVC |
|---|---|---|
| Parametri ottimizzati | sì (ansatz) | no |
| Tipo di problema | non convesso | convesso (SVM duale) |
| Costo training | O(N) iterazioni × profondità | O(N²) coppie per la matrice kernel |
| Ipotesi sulla loss | gradient-free (COBYLA) | quadratica (libsvm) |

Il QSVC ha il vantaggio di una loss convessa (niente local minima, niente barren plateaus), ma il difetto di scalare $O(N^2)$ in memoria e tempo per costruire la matrice kernel.

### 13.1 Perché un sample size ridotto

Con `SAMPLE_SIZE = 2000` (di cui 1600 in training) la matrice kernel sarebbe $1600 \times 1600 = 2.56 \cdot 10^6$ entries. Ogni entry richiede l'esecuzione di un circuito quantistico: anche su simulatore statevector veloce, sono molte ore di calcolo, e la memoria può saturare facilmente (specialmente con le vecchie implementazioni che costruivano internamente un `probabilities_dict` di taglia $2^{n}$ per ogni circuito).

Inoltre usiamo `FidelityStatevectorKernel` invece del più vecchio `FidelityQuantumKernel`: la versione *statevector* lavora direttamente sui vettori di stato senza mai costruire dizionari di probabilità, ed è molto più leggera in memoria su >= 8 qubit.

In [ ]:
# Import con fallback per compatibilità con vecchie versioni di qiskit-ml
try:
    from qiskit_machine_learning.kernels import FidelityStatevectorKernel
    _KERNEL_CLS = FidelityStatevectorKernel
    _KERNEL_NAME = "FidelityStatevectorKernel"
except ImportError:
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    _KERNEL_CLS = FidelityQuantumKernel
    _KERNEL_NAME = "FidelityQuantumKernel (fallback)"

from qiskit_machine_learning.algorithms import QSVC

print(f"Kernel class : {_KERNEL_NAME}")

### 13.2 Subset dedicato al QSVC

Per non penalizzare il QSVC con un dataset diverso da quello usato per RF e VQC, ripartiamo dai dati *già pre-processati* (post StandardScaler → feature selection → PCA → MinMaxScaler $[0, \pi]$).

Il modo più pulito sarebbe rifare la pipeline da zero. Per semplicità qui partiamo direttamente dall'`X` PCA-ridotto (ricordiamo che `X` nel notebook è già il dataset post-PCA su 8 componenti) e applichiamo lo scaling quantistico.

In [ ]:

# *già* PCA-ridotto (variabile X). y è coerente.
X_q_full, y_q = balanced_subsample(X, y, SAMPLE_SIZE, seed=RANDOM_SEED)

# Riapplichiamo lo stesso rescaling [0, pi] usato per i VQC
qsvc_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_q = qsvc_scaler.fit_transform(X_q_full)

Xq_tr_s, Xq_te_s, yq_tr_s, yq_te_s = train_test_split(
    X_q, y_q,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_q,
)
print(f"QSVC train : {Xq_tr_s.shape} | test : {Xq_te_s.shape}")

### 13.3 Costruzione e training del QSVC

Costruiamo il `QSVC` passandogli il quantum kernel basato sulla nostra feature map. Sotto il cofano, `QSVC` eredita da `sklearn.svm.SVC` e usa libsvm per risolvere il problema duale: la matrice di Gram è però costruita con il kernel quantistico, una entry alla volta tramite il simulatore.

In [ ]:
qkernel = _KERNEL_CLS(feature_map=feature_map)
qsvc    = QSVC(quantum_kernel=qkernel)

try:
    results.append(
        evaluate_model(qsvc, Xq_tr_s, yq_tr_s, Xq_te_s, yq_te_s,
                       "QSVC (ZZ kernel)", is_quantum=True)
    )
except MemoryError as e:
    print(f"!! QSVC fallito per memoria: {e}")
    print(f"!! Riduci QSVC_SAMPLE_SIZE o N_COMPONENTS e riprova.")
except Exception as e:
    print(f"!! QSVC fallito: {type(e).__name__}: {e}")

## 14. Confronto finale tra tutti i modelli

Stampiamo il ranking completo per F1-score ed esportiamo la tabella in CSV. Questa è la tabella che andrà inclusa nel capitolo di valutazione della tesi.

In [ ]:
# Tabella ordinata per F1 decrescente
header = (f"{'Modello':<26}{'Tipo':<11}"
          f"{'Pr':>7}{'Rc':>7}{'F1':>7}{'TestAcc':>9}{'Time(s)':>10}")
print(header)
print("-" * len(header))
for r in sorted(results, key=lambda r: r["f1"], reverse=True):
    print(f"{r['name']:<26}{r['type']:<11}"
          f"{r['precision']:>7.3f}{r['recall']:>7.3f}{r['f1']:>7.3f}"
          f"{r['test_acc']:>9.3f}{r['train_time']:>10.1f}")

# Esporta CSV finale
out_csv = "rf_vs_vqc_vs_qsvc_results.csv"
pd.DataFrame(results).to_csv(out_csv, index=False, float_format="%.6f")
print(f"\nRisultati esportati in: {out_csv}")

## 15. Conclusioni complessive

Abbiamo confrontato sullo stesso split train/test del dataset CICIDS2017 quattro modelli: il Random Forest classico (sia su 20 feature selezionate che sulle 8 componenti PCA), due VQC con ansatz diversi e un QSVC con quantum kernel.

Le osservazioni principali sono:

- **Il Random Forest resta il modello di riferimento.** Su CICIDS2017 raggiunge F1 > 0.95 anche con sole 8 componenti PCA, in tempi di addestramento dell'ordine del secondo. È molto difficile da battere su questo tipo di dato tabellare.
- **Il VQC con `efficient_su2` tipicamente supera quello con `real_amplitudes`**: la maggiore espressività dell'ansatz $R_y + R_z$ giustifica il costo del raddoppio dei parametri. Nessuno dei due raggiunge però le performance del Random Forest sullo stesso input.
- **Il QSVC è competitivo in accuracy ma costoso.** La loss convessa del kernel method evita i problemi di ottimizzazione del VQC.
- **Tempi di training**: RF ≈ 1 s, VQC ≈ minuti, QSVC ≈ secondi. Il gap di efficienza tra classico e quantistico simulato è ancora di 2-3 ordini di grandezza.

Considerazioni per il progetto:

- L'esperimento conferma che, **su simulatore classico**, il quantum ML non compete con i metodi classici su CICIDS2017 in termini né di accuracy né di tempo. Questo è coerente con la letteratura attuale sul QML applicato all'intrusion detection.
- Il valore dell'esperimento è documentare lo stato dell'arte e quantificare il gap, fornendo una baseline riproducibile per futuri lavori che usino hardware quantistico reale (dove il QSVC potrebbe trarre vantaggio dall'esecuzione parallela del kernel) o feature map più sofisticate.

